# CNN Baseline Diagnostic Tests

## Purpose and scope

This notebook diagnoses why the image-level CNN baseline stays near chance accuracy on the 1-bit Clean versus LSB-Manipulated task. It implements the four diagnostics defined in `04_Outputs/cnn_baseline_tests_task.md` following `04_Outputs/CV_test/AAI3001-CNN-Baseline-Diagnostic-Implementation-Plan.md`.

This notebook is for testing and diagnosis only. It does not improve the classifier, it is not `cnn_baselineV2`, and it does not use the held-out test set for training, tuning, or model selection.

## How to run

Run all cells from a fresh kernel, top to bottom, with the local virtual environment kernel that has PyTorch. Each test prints its result and saves a JSON record under `diagnostic_results/` before the next test starts. Fill each observation cell after its test completes.

## Recorded environment and reproducibility corrections

- The notebook runs on the local machine and uses the local data root.
- The baseline `cnn_baseline.ipynb` defines `train_epoch` twice, and its later definition refers to an unrelated `loader` variable in a progress-bar statement. This notebook defines `train_epoch` once and does not depend on variables left in a Jupyter kernel.
- The model, optimiser, and split follow the existing baseline. The test manifest is read only for its split size, never for images or training.
- Batch size 64 needs roughly 1.5 GB of GPU memory. If CUDA runs out of memory, reduce `BATCH_SIZE` and keep it equal across the compared runs.

## Setup and configuration

In [2]:
import json
import random
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

RANDOM_SEED = 42
BATCH_SIZE = 64
LEARNING_RATE = 0.01
MOMENTUM = 0.9
NUM_EPOCHS = 3

TEST2_PER_CLASS = 16
TEST2_MAX_EPOCHS = 50
TEST2_TARGET_ACCURACY = 0.95
TEST2_SEED = 20260927

TEST3_TRAIN_SOURCES = 5000
TEST3_VAL_SOURCES = 1000
TEST3_SEED = 20260928

TEST4_TRAIN_SOURCES = 2000
TEST4_VAL_SOURCES = 500
TEST4_SEED = 20260929
TEST4_BITS = 4
TEST4_CONTROL_BITS = 1

DATA_ROOT = Path(
    r"C:\Users\zhiho\Desktop\AI_stuff\05_Projects\School"
    r"\Computer Vision and Deep Learning - Term Project"
    r"\05_Data"
)
MANIFEST_DIR = DATA_ROOT / "manifests"
RESULTS_DIR = DATA_ROOT.parent / "04_Outputs" / "CV_test" / "notebooks" / "diagnostic_results"
DIAGNOSTIC_DATA_DIR = DATA_ROOT / "diagnostic-4bit"

TRAIN_CSV = MANIFEST_DIR / "train_classifier.csv"
VAL_CSV = MANIFEST_DIR / "val_classifier.csv"
TEST_CSV = MANIFEST_DIR / "test_classifier.csv"
SOURCES_CSV = MANIFEST_DIR / "sources.csv"

SOURCE_DTYPE_MAP = {"source_id": str, "generator_source_id": str}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Shared definitions

In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class LSBClassificationDataset(Dataset):
    def __init__(self, dataframe, data_root, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.data_root = Path(data_root)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = self.data_root / row["image_path"]

        with Image.open(image_path) as image_file:
            image = image_file.convert("RGB")

        label = int(row["label"])
        if self.transform is not None:
            image = self.transform(image)

        return image, label


def make_train_transform():
    orientation_augmentations = transforms.RandomChoice([
        transforms.RandomRotation(degrees=(90, 90)),
        transforms.RandomRotation(degrees=(180, 180)),
        transforms.RandomRotation(degrees=(270, 270)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.RandomVerticalFlip(p=1.0),
    ])
    return transforms.Compose([
        orientation_augmentations,
        transforms.ToTensor(),
    ])


def make_eval_transform():
    return transforms.Compose([
        transforms.ToTensor(),
    ])


def load_split_manifests():
    train = pd.read_csv(TRAIN_CSV, dtype=SOURCE_DTYPE_MAP)
    val = pd.read_csv(VAL_CSV, dtype=SOURCE_DTYPE_MAP)
    test = pd.read_csv(TEST_CSV, dtype=SOURCE_DTYPE_MAP)
    return train, val, test


def select_source_subset(dataframe, max_sources, seed):
    if max_sources <= 0 or max_sources >= dataframe["source_id"].nunique():
        return dataframe.reset_index(drop=True)

    source_ids = dataframe["source_id"].drop_duplicates().to_numpy()
    rng = np.random.default_rng(seed)
    chosen = set(rng.choice(source_ids, size=max_sources, replace=False).tolist())
    return dataframe[dataframe["source_id"].isin(chosen)].reset_index(drop=True)


def prediction_counts(predictions):
    return {
        "clean": int((predictions == 0).sum()),
        "stego": int((predictions == 1).sum()),
    }


def save_result(name, result):
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    path = RESULTS_DIR / f"{name}.json"
    with open(path, "w", encoding="utf-8") as result_file:
        json.dump(result, result_file, indent=2)
    print(f"Saved result to {path}")

In [4]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + self.shortcut(x)
        out = self.relu(out)
        return out


class CNNBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.residual_block1 = ResidualBlock(3, 16, stride=1)
        self.residual_block2 = ResidualBlock(16, 16, stride=1)
        self.residual_block3 = ResidualBlock(16, 16, stride=1)

        self.residual_block4 = ResidualBlock(16, 32, stride=2)
        self.residual_block5 = ResidualBlock(32, 32, stride=1)

        self.residual_block6 = ResidualBlock(32, 64, stride=2)
        self.residual_block7 = ResidualBlock(64, 64, stride=1)
        self.residual_block8 = ResidualBlock(64, 64, stride=1)

        self.fc1 = nn.Linear(64 * 32 * 32, 64)
        self.fc2 = nn.Linear(64, 2)
        self.relu = nn.ReLU()

    def forward(self, images):
        images = self.residual_block1(images)
        images = self.residual_block2(images)
        images = self.residual_block3(images)
        images = self.residual_block4(images)
        images = self.residual_block5(images)
        images = self.residual_block6(images)
        images = self.residual_block7(images)
        images = self.residual_block8(images)

        images = images.view(images.size(0), -1)
        images = self.fc1(images)
        images = self.relu(images)
        images = self.fc2(images)
        return images


def build_cnn_baseline():
    return CNNBaseline()


def make_criterion():
    return nn.CrossEntropyLoss()


def make_optimizer(model):
    return optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)

In [5]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            predictions = outputs.argmax(dim=1)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (predictions == labels).sum().item()
            total_samples += batch_size
            all_labels.append(labels.cpu())
            all_predictions.append(predictions.cpu())

    return (
        total_loss / total_samples,
        total_correct / total_samples,
        torch.cat(all_labels),
        torch.cat(all_predictions),
    )


def train_classifier(run_name, train_subset, val_subset, train_transform, seed, epochs=NUM_EPOCHS):
    set_seed(seed)

    model = build_cnn_baseline().to(device)
    criterion = make_criterion()
    optimizer = make_optimizer(model)

    train_loader = DataLoader(
        LSBClassificationDataset(train_subset, DATA_ROOT, train_transform),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(
        LSBClassificationDataset(val_subset, DATA_ROOT, make_eval_transform()),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    epoch_results = []
    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_accuracy, _, val_predictions = evaluate(model, val_loader, criterion, device)

        epoch_results.append({
            "epoch": epoch,
            "train_loss": round(train_loss, 6),
            "train_accuracy": round(train_accuracy, 6),
            "val_loss": round(val_loss, 6),
            "val_accuracy": round(val_accuracy, 6),
            "val_prediction_counts": prediction_counts(val_predictions),
        })

        print(
            f"{run_name} | epoch {epoch}/{epochs} | "
            f"train loss {train_loss:.4f} | train accuracy {train_accuracy:.4f} | "
            f"val loss {val_loss:.4f} | val accuracy {val_accuracy:.4f}"
        )

    return {
        "run_name": run_name,
        "train_rows": int(len(train_subset)),
        "val_rows": int(len(val_subset)),
        "epochs": epoch_results,
        "final_train_accuracy": epoch_results[-1]["train_accuracy"],
        "final_val_accuracy": epoch_results[-1]["val_accuracy"],
        "val_prediction_counts": epoch_results[-1]["val_prediction_counts"],
    }

## Environment record

In [6]:
train_df, val_df, test_df = load_split_manifests()
sources_df = pd.read_csv(SOURCES_CSV, dtype=SOURCE_DTYPE_MAP)

run_folder = Path(train_df.loc[0, "image_path"]).parts[0]
with Image.open(DATA_ROOT / train_df.loc[0, "image_path"]) as image_file:
    reference_image = image_file.convert("RGB")
    reference_image_size = reference_image.size

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Device:", device)
print("Random seed:", RANDOM_SEED)
print("Data root:", DATA_ROOT)
print("Manifest paths:")
print(" -", TRAIN_CSV)
print(" -", VAL_CSV)
print(" -", TEST_CSV)
print("Data run folder:", run_folder)
print("Reference image size:", reference_image_size)
print("Model: CNNBaseline, 8 residual blocks, flatten, Linear(64 * 32 * 32, 64), ReLU, Linear(64, 2)")
print("Loss: cross-entropy")
print("Optimiser: SGD, learning rate", LEARNING_RATE, "momentum", MOMENTUM)
print("Batch size:", BATCH_SIZE)
print("Epoch limit:", NUM_EPOCHS)
print("Training transform: one orientation augmentation (90, 180, or 270 degree rotation, horizontal flip, or vertical flip), then ToTensor")
print("Evaluation transform: ToTensor")
print("Splits:")
print(f" train rows {len(train_df)} from {train_df['source_id'].nunique()} sources")
print(f" validation rows {len(val_df)} from {val_df['source_id'].nunique()} sources")
print(f" test rows {len(test_df)} from {test_df['source_id'].nunique()} sources")

Python: 3.13.7
PyTorch: 2.14.0+cu126 | CUDA build: 12.6 | CUDA available: True
GPU: NVIDIA GeForce MX250
Device: cuda
Random seed: 42
Data root: C:\Users\zhiho\Desktop\AI_stuff\05_Projects\School\Computer Vision and Deep Learning - Term Project\05_Data
Manifest paths:
 - C:\Users\zhiho\Desktop\AI_stuff\05_Projects\School\Computer Vision and Deep Learning - Term Project\05_Data\manifests\train_classifier.csv
 - C:\Users\zhiho\Desktop\AI_stuff\05_Projects\School\Computer Vision and Deep Learning - Term Project\05_Data\manifests\val_classifier.csv
 - C:\Users\zhiho\Desktop\AI_stuff\05_Projects\School\Computer Vision and Deep Learning - Term Project\05_Data\manifests\test_classifier.csv
Data run folder: lsb-run-20260922-233255-728056
Reference image size: (128, 128)
Model: CNNBaseline, 8 residual blocks, flatten, Linear(64 * 32 * 32, 64), ReLU, Linear(64, 2)
Loss: cross-entropy
Optimiser: SGD, learning rate 0.01 momentum 0.9
Batch size: 64
Epoch limit: 3
Training transform: one orientation

## Test 1. Dataset and label sanity

Purpose. Confirm that the classifier data loads correctly before any model result is interpreted. This test also checks label values, class names, folder and label agreement, file existence, fixed-seed sample images, and source separation between splits. The held-out test manifest is read for its count and source identities only.

In [6]:
print("Test 1. Dataset and label sanity")
print()

split_summary_rows = []
for split_name, split_df in (("train", train_df), ("validation", val_df), ("test", test_df)):
    total = len(split_df)
    clean_count = int(split_df["label"].eq(0).sum())
    stego_count = int(split_df["label"].eq(1).sum())
    split_summary_rows.append({
        "split": split_name,
        "rows": total,
        "clean": clean_count,
        "stego": stego_count,
        "clean_percent": round(100 * clean_count / total, 2),
        "stego_percent": round(100 * stego_count / total, 2),
    })
split_summary = pd.DataFrame(split_summary_rows)
print(split_summary.to_string(index=False))
print()

label_to_class = (
    pd.concat([train_df[["label", "class_name"]], val_df[["label", "class_name"]]])
    .drop_duplicates()
    .sort_values("label")
)
print("Label and class values:")
print(label_to_class.to_string(index=False))
print()


def label_folder_mismatches(split_df):
    folders = split_df["image_path"].str.split("/").str[-2]
    expected_folders = split_df["label"].map({0: "clean", 1: "stego"})
    return split_df[folders.ne(expected_folders)]


train_mismatches = label_folder_mismatches(train_df)
val_mismatches = label_folder_mismatches(val_df)
print("Rows whose image folder does not match the label folder:", len(train_mismatches), "train,", len(val_mismatches), "validation")
print()

print(f"Checking that every referenced train and validation image exists ({len(train_df) + len(val_df)} files)...")
missing_files = [
    image_path
    for image_path in pd.concat([train_df["image_path"], val_df["image_path"]])
    if not (DATA_ROOT / image_path).is_file()
]
print("Missing files:", len(missing_files))
print()

sample_rng = np.random.default_rng(RANDOM_SEED)
sample_rows = []
for split_name, split_df in (("train", train_df), ("validation", val_df)):
    positions = sample_rng.choice(split_df.index.to_numpy(), size=3, replace=False)
    for position in positions:
        row = split_df.loc[position]
        with Image.open(DATA_ROOT / row["image_path"]) as image_file:
            image = image_file.convert("RGB")
            tensor = make_eval_transform()(image)
        sample_rows.append({
            "split": split_name,
            "source_id": row["source_id"],
            "label": int(row["label"]),
            "class_name": row["class_name"],
            "image_path": row["image_path"],
            "mode": image.mode,
            "size": str(image.size),
            "tensor_shape": str(tuple(tensor.shape)),
            "tensor_dtype": str(tensor.dtype),
            "tensor_min": round(float(tensor.min()), 4),
            "tensor_max": round(float(tensor.max()), 4),
        })
print("Fixed-seed sample image and label pairs:")
print(pd.DataFrame(sample_rows).to_string(index=False))
print()

train_sources = set(train_df["source_id"])
val_sources = set(val_df["source_id"])
test_sources = set(test_df["source_id"])
source_overlaps = {
    "train_val": len(train_sources & val_sources),
    "train_test": len(train_sources & test_sources),
    "val_test": len(val_sources & test_sources),
}
print("Source identity overlaps between splits:", source_overlaps)
print()

label_class_ok = label_to_class.set_index("label")["class_name"].to_dict() == {0: "clean", 1: "stego"}

test1_problems = []
if not label_class_ok:
    test1_problems.append("labels are not exactly 0 for clean and 1 for stego")
if missing_files:
    test1_problems.append(f"{len(missing_files)} referenced train or validation images are missing, first {missing_files[0]}")
if len(train_mismatches) or len(val_mismatches):
    test1_problems.append(
        f"{len(train_mismatches)} train and {len(val_mismatches)} validation rows have an image folder that does not match the label"
    )
if any(source_overlaps.values()):
    test1_problems.append(f"source identities cross splits: {source_overlaps}")

test1_result = {
    "status": "passed" if not test1_problems else "failed",
    "problems": test1_problems,
    "split_summary": split_summary_rows,
    "label_values": sorted(int(value) for value in set(train_df["label"]) | set(val_df["label"])),
    "class_names": sorted(set(train_df["class_name"]) | set(val_df["class_name"])),
    "run_folder": run_folder,
    "reference_image_size": list(reference_image_size),
    "sample_pairs": sample_rows,
    "missing_files": len(missing_files),
    "missing_file_examples": missing_files[:5],
    "train_label_folder_mismatches": int(len(train_mismatches)),
    "val_label_folder_mismatches": int(len(val_mismatches)),
    "source_overlaps": source_overlaps,
}
save_result("test1_dataset_sanity", test1_result)

if test1_problems:
    raise RuntimeError(
        "Test 1 found problems that must be corrected before model results are interpreted: "
        + "; ".join(test1_problems)
    )

print("Test 1 passed. Continue to Test 2.")

Test 1. Dataset and label sanity

     split   rows  clean  stego  clean_percent  stego_percent
     train 100000  50000  50000           50.0           50.0
validation  20000  10000  10000           50.0           50.0
      test  20000  10000  10000           50.0           50.0

Label and class values:
 label class_name
     0      clean
     1      stego

Rows whose image folder does not match the label folder: 0 train, 0 validation

Checking that every referenced train and validation image exists (120000 files)...
Missing files: 0

Fixed-seed sample image and label pairs:
     split source_id  label class_name                                      image_path mode       size  tensor_shape  tensor_dtype  tensor_min  tensor_max
     train     45831      1      stego lsb-run-20260922-233255-728056/stego/045832.png  RGB (128, 128) (3, 128, 128) torch.float32      0.0824         1.0
     train     06187      0      clean lsb-run-20260922-233255-728056/clean/006188.png  RGB (128, 128) (3,

### Test 1 observation (record after running)

Record what the counts, sample pairs, and checks show. If Test 1 failed, record the problem and the correction before continuing.

## Test 2. Small-subset overfit

Purpose. Test whether the existing model and training pipeline can memorise a fixed, small, balanced subset taken from the training split only. Augmentation is disabled so every example stays fixed, and the baseline SGD setting is kept. With 32 examples and batch size 64 there is one optimiser step per epoch.

In [7]:
print("Test 2. Small-subset overfit")
print()

subset_rng = np.random.default_rng(TEST2_SEED)
clean_positions = train_df.index[train_df["label"].eq(0)].to_numpy()
stego_positions = train_df.index[train_df["label"].eq(1)].to_numpy()
subset_positions = np.concatenate([
    subset_rng.choice(clean_positions, size=TEST2_PER_CLASS, replace=False),
    subset_rng.choice(stego_positions, size=TEST2_PER_CLASS, replace=False),
])
test2_df = train_df.loc[subset_positions].reset_index(drop=True)

print("Fixed subset rows:", len(test2_df))
print(test2_df["class_name"].value_counts().to_string())
print("First rows of the fixed subset:")
print(test2_df[["source_id", "label", "class_name", "image_path"]].head(8).to_string(index=False))
print("Augmentation disabled, so each example is fixed across epochs.")
print()

set_seed(TEST2_SEED)
model = build_cnn_baseline().to(device)
criterion = make_criterion()
optimizer = make_optimizer(model)

subset_loader = DataLoader(
    LSBClassificationDataset(test2_df, DATA_ROOT, make_eval_transform()),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

epoch_rows = []
for epoch in range(1, TEST2_MAX_EPOCHS + 1):
    train_loss, train_accuracy = train_epoch(model, subset_loader, criterion, optimizer, device)
    epoch_rows.append({
        "epoch": epoch,
        "train_loss": round(train_loss, 6),
        "train_accuracy": round(train_accuracy, 6),
    })
    print(f"Epoch {epoch:02d} | train loss: {train_loss:.4f} | train accuracy: {train_accuracy:.4f}")
    if train_accuracy >= TEST2_TARGET_ACCURACY:
        print(f"Stopping early because training accuracy reached {TEST2_TARGET_ACCURACY:.2f}.")
        break

_, _, _, test2_predictions = evaluate(model, subset_loader, criterion, device)
test2_counts = prediction_counts(test2_predictions)
memorised = epoch_rows[-1]["train_accuracy"] >= TEST2_TARGET_ACCURACY

print()
print("Final train accuracy:", round(epoch_rows[-1]["train_accuracy"], 4))
print("Optimiser steps:", len(epoch_rows))
print("Prediction counts on the fixed subset:", test2_counts)

test2_result = {
    "subset_rows": int(len(test2_df)),
    "subset_class_counts": {name: int(count) for name, count in test2_df["class_name"].value_counts().items()},
    "subset_source_ids": test2_df["source_id"].tolist(),
    "max_epochs": TEST2_MAX_EPOCHS,
    "target_accuracy": TEST2_TARGET_ACCURACY,
    "optimiser_steps": len(epoch_rows),
    "epochs": epoch_rows,
    "memorised": bool(memorised),
    "prediction_counts": test2_counts,
}
save_result("test2_overfit_subset", test2_result)

Test 2. Small-subset overfit

Fixed subset rows: 32
class_name
clean    16
stego    16
First rows of the fixed subset:
source_id  label class_name                                      image_path
    48224      0      clean lsb-run-20260922-233255-728056/clean/048225.png
    28466      0      clean lsb-run-20260922-233255-728056/clean/028467.png
    62797      0      clean lsb-run-20260922-233255-728056/clean/062798.png
    01743      0      clean lsb-run-20260922-233255-728056/clean/001744.png
    39333      0      clean lsb-run-20260922-233255-728056/clean/039334.png
    17183      0      clean lsb-run-20260922-233255-728056/clean/017184.png
    03246      0      clean lsb-run-20260922-233255-728056/clean/003247.png
    65608      0      clean lsb-run-20260922-233255-728056/clean/065609.png
Augmentation disabled, so each example is fixed across epochs.

Epoch 01 | train loss: 0.6853 | train accuracy: 0.5938
Epoch 02 | train loss: 3.7057 | train accuracy: 0.5000
Epoch 03 | train loss: 

### Test 2 observation (record after running)

Record whether training accuracy rose substantially above random guessing, how many epochs and optimiser steps it took, and the final prediction distribution. If the subset was not memorised, note the pipeline issue that the evidence points to before attributing the failure to a weak 1-bit signal.

## Test 3. Training without augmentation

Purpose. Compare a local augmented control against a no-augmentation run on the same data, seed, model, optimiser, batch size, and epoch count. Only the training orientation transform changes. The reported baseline behaviour near 50 percent accuracy is historical context. The local control is the fair comparison for this test.

Data. The default configuration uses a source-safe subset because a full local run is impractical on the local GPU. Set `TEST3_TRAIN_SOURCES` or `TEST3_VAL_SOURCES` to 0 to run the full split. A subset result is narrower than a full-split result, and the notebook states which one ran.

In [8]:
print("Test 3. Training without augmentation")
print()

test3_train_df = select_source_subset(train_df, TEST3_TRAIN_SOURCES, TEST3_SEED)
test3_val_df = select_source_subset(val_df, TEST3_VAL_SOURCES, TEST3_SEED)

print(
    f"Train rows {len(test3_train_df)} from {test3_train_df['source_id'].nunique()} sources | "
    f"validation rows {len(test3_val_df)} from {test3_val_df['source_id'].nunique()} sources"
)
if len(test3_train_df) < len(train_df):
    print("A source-safe subset is in use. The conclusion is narrower than a full-split run.")
print()

augmented_condition = train_classifier(
    "augmented", test3_train_df, test3_val_df, make_train_transform(), TEST3_SEED
)
no_augmentation_condition = train_classifier(
    "no augmentation", test3_train_df, test3_val_df, make_eval_transform(), TEST3_SEED
)

comparison_rows = []
for condition_name, condition in (
    ("augmented", augmented_condition),
    ("no augmentation", no_augmentation_condition),
):
    comparison_rows.append({
        "condition": condition_name,
        "epochs_run": len(condition["epochs"]),
        "final_train_loss": condition["epochs"][-1]["train_loss"],
        "final_train_accuracy": condition["final_train_accuracy"],
        "final_val_loss": condition["epochs"][-1]["val_loss"],
        "final_val_accuracy": condition["final_val_accuracy"],
        "val_clean_predictions": condition["val_prediction_counts"]["clean"],
        "val_stego_predictions": condition["val_prediction_counts"]["stego"],
    })

print()
print("Condition comparison:")
print(pd.DataFrame(comparison_rows).to_string(index=False))

test3_result = {
    "train_rows": int(len(test3_train_df)),
    "val_rows": int(len(test3_val_df)),
    "train_sources": int(test3_train_df["source_id"].nunique()),
    "val_sources": int(test3_val_df["source_id"].nunique()),
    "subset_used": bool(len(test3_train_df) < len(train_df)),
    "augmented": augmented_condition,
    "no_augmentation": no_augmentation_condition,
}
save_result("test3_augmentation_comparison", test3_result)

Test 3. Training without augmentation

Train rows 10000 from 5000 sources | validation rows 2000 from 1000 sources
A source-safe subset is in use. The conclusion is narrower than a full-split run.

augmented | epoch 1/3 | train loss 0.8152 | train accuracy 0.5010 | val loss 0.6944 | val accuracy 0.5000
augmented | epoch 2/3 | train loss 0.6935 | train accuracy 0.4968 | val loss 0.6932 | val accuracy 0.5000
augmented | epoch 3/3 | train loss 0.6934 | train accuracy 0.4995 | val loss 0.6932 | val accuracy 0.5000
no augmentation | epoch 1/3 | train loss 0.7736 | train accuracy 0.5012 | val loss 0.6944 | val accuracy 0.5000
no augmentation | epoch 2/3 | train loss 0.6935 | train accuracy 0.5006 | val loss 0.6934 | val accuracy 0.5000
no augmentation | epoch 3/3 | train loss 0.6934 | train accuracy 0.4938 | val loss 0.6932 | val accuracy 0.5000

Condition comparison:
      condition  epochs_run  final_train_loss  final_train_accuracy  final_val_loss  final_val_accuracy  val_clean_prediction

### Test 3 observation (record after running)

Record whether the local augmented control reproduces the reported near-chance behaviour, whether removing augmentation changes training or validation behaviour, and the validation prediction distribution for each condition. Note that a subset run gives a narrower conclusion than a full-split run.

## Test 4. Stronger-stego sanity

Purpose. Test whether the same pipeline learns when the embedding is stronger. The project generator writes only the lowest bit, so this test prepares a separate diagnostic dataset from the same clean sources. Nothing here changes the project generator or the held-out test set.

Embedding rule. For every source, the selected fraction of all RGB channel slots is the recorded per-source payload rate. The 4-bit condition replaces the lowest four bits of each selected slot with a seeded payload value. The matched 1-bit control replaces the lowest bit with the lowest bit of the same seeded value at the same slots. Selection and payloads are seeded from the source id, so the 4-bit images and the 1-bit control are matched and the generation is reproducible.

Training. Both depth conditions use the same model, training transform, split, optimiser, and epoch count, and both start from the same seed.

In [10]:
print("Test 4. Stronger-stego sanity")
print()


def select_diagnostic_sources(split_name, max_sources, seed):
    split_sources = sources_df[sources_df["split"].eq(split_name)].reset_index(drop=True)
    if 0 < max_sources < len(split_sources):
        rng = np.random.default_rng(seed)
        chosen = rng.choice(split_sources.index.to_numpy(), size=max_sources, replace=False)
        split_sources = split_sources.loc[chosen].reset_index(drop=True)
    return split_sources


def build_stego_pair(clean_array, payload_rate, seed):
    rng = np.random.default_rng(seed)
    flat = clean_array.reshape(-1)
    position_count = int(round(payload_rate * flat.size))
    positions = rng.choice(flat.size, size=position_count, replace=False)
    payload_four_bit = rng.integers(0, 1 << TEST4_BITS, size=position_count, dtype=np.uint8)

    stego_four_bit = clean_array.copy()
    stego_one_bit = clean_array.copy()
    flat_four_bit = stego_four_bit.reshape(-1)
    flat_one_bit = stego_one_bit.reshape(-1)

    keep_mask = (0xFF << TEST4_BITS) & 0xFF
    control_keep_mask = (0xFF << TEST4_CONTROL_BITS) & 0xFF
    control_value_mask = (1 << TEST4_CONTROL_BITS) - 1
    flat_four_bit[positions] = (flat_four_bit[positions] & keep_mask) | payload_four_bit
    flat_one_bit[positions] = (
        (flat_one_bit[positions] & control_keep_mask) | (payload_four_bit & control_value_mask)
    )

    return stego_four_bit, stego_one_bit


def build_diagnostic_manifest(source_rows, stego_folder):
    records = []
    for _, row in source_rows.iterrows():
        source_id = row["source_id"]
        records.append({
            "source_id": source_id,
            "generator_source_id": row["generator_source_id"],
            "image_path": f"diagnostic-4bit/clean/{source_id}.png",
            "label": 0,
            "class_name": "clean",
            "payload_rate": 0.0,
        })
        records.append({
            "source_id": source_id,
            "generator_source_id": row["generator_source_id"],
            "image_path": f"diagnostic-4bit/{stego_folder}/{source_id}.png",
            "label": 1,
            "class_name": "stego",
            "payload_rate": float(row["payload_rate"]),
        })
    return pd.DataFrame(records)


train_4bit_sources = select_diagnostic_sources("train", TEST4_TRAIN_SOURCES, TEST4_SEED)
val_4bit_sources = select_diagnostic_sources("val", TEST4_VAL_SOURCES, TEST4_SEED)
print("Diagnostic sources:", len(train_4bit_sources), "train,", len(val_4bit_sources), "validation")
print("Diagnostic data directory:", DIAGNOSTIC_DATA_DIR)

clean_dir = DIAGNOSTIC_DATA_DIR / "clean"
stego_4bit_dir = DIAGNOSTIC_DATA_DIR / "stego-4bit"
stego_1bit_dir = DIAGNOSTIC_DATA_DIR / "stego-1bit"
for directory in (clean_dir, stego_4bit_dir, stego_1bit_dir):
    directory.mkdir(parents=True, exist_ok=True)

for split_name, source_rows in (("train", train_4bit_sources), ("validation", val_4bit_sources)):
    for position, (_, row) in enumerate(source_rows.iterrows(), start=1):
        source_id = row["source_id"]
        clean_path = DATA_ROOT / row["clean_path"]
        with Image.open(clean_path) as clean_file:
            clean_array = np.array(clean_file.convert("RGB"))
        stego_four_bit, stego_one_bit = build_stego_pair(
            clean_array, float(row["payload_rate"]), TEST4_SEED + int(source_id)
        )
        shutil.copyfile(clean_path, clean_dir / f"{source_id}.png")
        Image.fromarray(stego_four_bit).save(stego_4bit_dir / f"{source_id}.png")
        Image.fromarray(stego_one_bit).save(stego_1bit_dir / f"{source_id}.png")
        if position % 250 == 0 or position == len(source_rows):
            print(f"{split_name}: encoded {position}/{len(source_rows)} sources")

train_4bit_manifest = build_diagnostic_manifest(train_4bit_sources, "stego-4bit")
val_4bit_manifest = build_diagnostic_manifest(val_4bit_sources, "stego-4bit")
train_1bit_manifest = build_diagnostic_manifest(train_4bit_sources, "stego-1bit")
val_1bit_manifest = build_diagnostic_manifest(val_4bit_sources, "stego-1bit")

train_4bit_manifest.to_csv(DIAGNOSTIC_DATA_DIR / "train_4bit.csv", index=False)
val_4bit_manifest.to_csv(DIAGNOSTIC_DATA_DIR / "val_4bit.csv", index=False)
train_1bit_manifest.to_csv(DIAGNOSTIC_DATA_DIR / "train_1bit.csv", index=False)
val_1bit_manifest.to_csv(DIAGNOSTIC_DATA_DIR / "val_1bit.csv", index=False)

generation_config = {
    "seed": TEST4_SEED,
    "four_bit_depth": TEST4_BITS,
    "control_depth": TEST4_CONTROL_BITS,
    "selection": "uniform over all H*W*3 RGB channel slots, count is round(payload_rate * total slots)",
    "payload": "seeded uniform values, the matched control reuses the lowest bit of the same value",
    "train_sources": train_4bit_manifest["source_id"].drop_duplicates().tolist(),
    "val_sources": val_4bit_manifest["source_id"].drop_duplicates().tolist(),
}
with open(DIAGNOSTIC_DATA_DIR / "generation_config.json", "w", encoding="utf-8") as config_file:
    json.dump(generation_config, config_file, indent=2)

print("Diagnostic manifests written:")
for manifest_name in ("train_4bit.csv", "val_4bit.csv", "train_1bit.csv", "val_1bit.csv"):
    print(" -", DIAGNOSTIC_DATA_DIR / manifest_name)

Test 4. Stronger-stego sanity

Diagnostic sources: 2000 train, 500 validation
Diagnostic data directory: C:\Users\zhiho\Desktop\AI_stuff\05_Projects\School\Computer Vision and Deep Learning - Term Project\05_Data\diagnostic-4bit
train: encoded 250/2000 sources
train: encoded 500/2000 sources
train: encoded 750/2000 sources
train: encoded 1000/2000 sources
train: encoded 1250/2000 sources
train: encoded 1500/2000 sources
train: encoded 1750/2000 sources
train: encoded 2000/2000 sources
validation: encoded 250/500 sources
validation: encoded 500/500 sources
Diagnostic manifests written:
 - C:\Users\zhiho\Desktop\AI_stuff\05_Projects\School\Computer Vision and Deep Learning - Term Project\05_Data\diagnostic-4bit\train_4bit.csv
 - C:\Users\zhiho\Desktop\AI_stuff\05_Projects\School\Computer Vision and Deep Learning - Term Project\05_Data\diagnostic-4bit\val_4bit.csv
 - C:\Users\zhiho\Desktop\AI_stuff\05_Projects\School\Computer Vision and Deep Learning - Term Project\05_Data\diagnostic-4bit

In [11]:
print("Training the 4-bit condition and the matched 1-bit control")
print()

test4_results = {}
for condition_name, train_manifest_name, val_manifest_name in (
    ("4-bit", "train_4bit.csv", "val_4bit.csv"),
    ("1-bit control", "train_1bit.csv", "val_1bit.csv"),
):
    train_condition_df = pd.read_csv(DIAGNOSTIC_DATA_DIR / train_manifest_name, dtype=SOURCE_DTYPE_MAP)
    val_condition_df = pd.read_csv(DIAGNOSTIC_DATA_DIR / val_manifest_name, dtype=SOURCE_DTYPE_MAP)
    print()
    print(
        f"Condition {condition_name} | train rows {len(train_condition_df)} | "
        f"validation rows {len(val_condition_df)}"
    )
    test4_results[condition_name] = train_classifier(
        condition_name,
        train_condition_df,
        val_condition_df,
        make_train_transform(),
        TEST4_SEED,
    )

comparison_rows = []
for condition_name, condition in test4_results.items():
    comparison_rows.append({
        "condition": condition_name,
        "final_train_loss": condition["epochs"][-1]["train_loss"],
        "final_train_accuracy": condition["final_train_accuracy"],
        "final_val_loss": condition["epochs"][-1]["val_loss"],
        "final_val_accuracy": condition["final_val_accuracy"],
        "val_clean_predictions": condition["val_prediction_counts"]["clean"],
        "val_stego_predictions": condition["val_prediction_counts"]["stego"],
    })

print()
print("Depth comparison:")
print(pd.DataFrame(comparison_rows).to_string(index=False))

test4_result = {
    "embedding_rule": (
        "seeded uniform selection over all H*W*3 RGB channel slots at the recorded per-source payload rate. "
        "The 4-bit condition replaces the lowest four bits with seeded payload values. "
        "The matched 1-bit control replaces the lowest bit with the lowest bit of the same seeded value at the same slots."
    ),
    "data_directory": str(DIAGNOSTIC_DATA_DIR),
    "train_sources": int(len(train_4bit_sources)),
    "val_sources": int(len(val_4bit_sources)),
    "conditions": test4_results,
}
save_result("test4_strength_sanity", test4_result)

Training the 4-bit condition and the matched 1-bit control


Condition 4-bit | train rows 4000 | validation rows 1000
4-bit | epoch 1/3 | train loss 0.8029 | train accuracy 0.5010 | val loss 0.6932 | val accuracy 0.5000
4-bit | epoch 2/3 | train loss 0.6935 | train accuracy 0.4968 | val loss 0.6932 | val accuracy 0.5000
4-bit | epoch 3/3 | train loss 0.6935 | train accuracy 0.4965 | val loss 0.6932 | val accuracy 0.5000

Condition 1-bit control | train rows 4000 | validation rows 1000
1-bit control | epoch 1/3 | train loss 0.7955 | train accuracy 0.5015 | val loss 0.6932 | val accuracy 0.5000
1-bit control | epoch 2/3 | train loss 0.6940 | train accuracy 0.4993 | val loss 0.6932 | val accuracy 0.5000
1-bit control | epoch 3/3 | train loss 0.6935 | train accuracy 0.4963 | val loss 0.6932 | val accuracy 0.5000

Depth comparison:
    condition  final_train_loss  final_train_accuracy  final_val_loss  final_val_accuracy  val_clean_predictions  val_stego_predictions
        4-bit          0.

### Test 4 observation (record after running)

Record whether the matched 4-bit condition learned where the matched 1-bit control did not, whether the difference is consistent across epochs, and the final prediction distributions. The matched design controls the selection and payload seeds, so an improvement is unlikely to come from location or encoding differences.

## Preliminary diagnosis (Tests 1 to 4)

Purpose. Combine the first four saved results into an evidence-ordered draft. This preliminary draft predates the paired overfit test and the pixel audit. The combined diagnosis after Tests 5 and 6 supersedes it.

In [13]:
def load_result(name):
    with open(RESULTS_DIR / f"{name}.json", encoding="utf-8") as result_file:
        return json.load(result_file)


test1_result = load_result("test1_dataset_sanity")
test2_result = load_result("test2_overfit_subset")
test3_result = load_result("test3_augmentation_comparison")
test4_result = load_result("test4_strength_sanity")

draft_findings = []

if test1_result["status"] == "passed":
    draft_findings.append("Test 1 passed. No loading, label, missing-file, or split problem was found, so the failure is not a confirmed data problem.")
else:
    draft_findings.append("Test 1 failed. Fix the data problem before using model results.")

if test2_result["memorised"]:
    draft_findings.append(
        f"Test 2 memorised the fixed {test2_result['subset_rows']}-image subset in "
        f"{test2_result['optimiser_steps']} optimiser steps, so the training loop, labels, and gradients work."
    )
    draft_findings.append(
        "The failure on the full split therefore points to generalisation or a weak 1-bit signal rather than a broken pipeline."
    )
else:
    draft_findings.append(
        f"Test 2 did not memorise the fixed {test2_result['subset_rows']}-image subset in "
        f"{test2_result['optimiser_steps']} optimiser steps, so the leading category is a model or training pipeline issue."
    )
    draft_findings.append(
        "Investigate the training loop, gradients, labels, and model execution before attributing the failure to a weak 1-bit signal."
    )

augmented_accuracy = test3_result["augmented"]["final_val_accuracy"]
no_augmentation_accuracy = test3_result["no_augmentation"]["final_val_accuracy"]
if no_augmentation_accuracy - augmented_accuracy >= 0.05:
    draft_findings.append(
        f"Removing orientation augmentation raised validation accuracy from "
        f"{augmented_accuracy:.4f} to {no_augmentation_accuracy:.4f}, so the augmentation transform is implicated."
    )
elif augmented_accuracy - no_augmentation_accuracy >= 0.05:
    draft_findings.append(
        f"Orientation augmentation raised validation accuracy from "
        f"{no_augmentation_accuracy:.4f} to {augmented_accuracy:.4f}, so augmentation is not the cause of the failure."
    )
else:
    draft_findings.append(
        f"Augmentation changed validation accuracy only from {augmented_accuracy:.4f} to "
        f"{no_augmentation_accuracy:.4f}, so it is not a leading explanation."
    )

four_bit_accuracy = test4_result["conditions"]["4-bit"]["final_val_accuracy"]
one_bit_accuracy = test4_result["conditions"]["1-bit control"]["final_val_accuracy"]
if four_bit_accuracy - one_bit_accuracy >= 0.05:
    draft_findings.append(
        f"The matched 4-bit condition reached {four_bit_accuracy:.4f} validation accuracy against "
        f"{one_bit_accuracy:.4f} for the matched 1-bit control, which supports the signal-strength hypothesis. "
        "The matched selection and payload design makes an encoding or location difference an unlikely explanation."
    )
else:
    draft_findings.append(
        f"The matched 4-bit condition reached {four_bit_accuracy:.4f} validation accuracy against "
        f"{one_bit_accuracy:.4f} for the matched 1-bit control, so the strength test did not separate the two conditions."
    )

if not test2_result["memorised"]:
    first_issue = "Test 2 blocks the earlier assumption. Diagnose and fix the model or training path before changing the input representation."
elif four_bit_accuracy - one_bit_accuracy >= 0.05:
    first_issue = "Stage 1 of cnn_baselineV2. Improve the input representation with residual or high-pass preprocessing, then re-test the 1-bit task."
else:
    first_issue = "Stage 1 of cnn_baselineV2. Compare the raw RGB input against a residual or high-pass representation as the first controlled change."

print("Draft diagnosis")
for finding in draft_findings:
    print("-", finding)
print()
print("Suggested first issue for cnn_baselineV2:", first_issue)

Draft diagnosis
- Test 1 passed. No loading, label, missing-file, or split problem was found, so the failure is not a confirmed data problem.
- Test 2 memorised the fixed 32-image subset in 21 optimiser steps, so the training loop, labels, and gradients work.
- The failure on the full split therefore points to generalisation or a weak 1-bit signal rather than a broken pipeline.
- Augmentation changed validation accuracy only from 0.5000 to 0.5000, so it is not a leading explanation.
- The matched 4-bit condition reached 0.5000 validation accuracy against 0.5000 for the matched 1-bit control, so the strength test did not separate the two conditions.

Suggested first issue for cnn_baselineV2: Stage 1 of cnn_baselineV2. Compare the raw RGB input against a residual or high-pass representation as the first controlled change.


### Preliminary diagnosis notes (Tests 1 to 4)

Confirm or amend the preliminary draft here. The combined diagnosis after Tests 5 and 6 supersedes these notes.

- Most likely failure category:
- Evidence:
- First issue to address in `cnn_baselineV2`:

## Test 5. Paired clean/stego overfit

Purpose. Test whether the unchanged CNN can fit matched clean and stego images from the same sources, which removes source-image identity as a shortcut. The stages use the first 1, 4, and 16 selected training sources, giving 2, 8, and 32 rows.

Method. Each stage starts from a fresh model and optimiser with the same seed, uses `ToTensor()` only, and keeps the baseline batch size of 64, so one optimiser step covers the whole subset. Training metrics come from the forward pass before that step's weight update. Evaluation-mode metrics come after the update. A stage counts as memorised only when evaluation accuracy is 1.0 and evaluation cross-entropy is below 0.1 for three consecutive checks. The step cap is 200.

Notes. If training-mode accuracy rises while evaluation-mode accuracy stays low, the next check is BatchNorm train and evaluation behaviour. Failing to fit a pair alone does not prove that raw RGB is insensitive to the LSB change. A 200-step cap is not proof that a larger subset cannot be fitted. Fill each stage observation before the next stage.

In [7]:
print("Test 5. Paired clean/stego overfit")
print("Preflight")
print()

TEST5_SEED = 20261001
TEST5_SOURCE_COUNT = 16
TEST5_MAX_STEPS = 200
TEST5_EVAL_LOSS_TARGET = 0.1
TEST5_CONSECUTIVE_CHECKS = 3
TEST5_SPIKE_THRESHOLD = 10.0

main_run_folder = Path(train_df.loc[0, "image_path"]).parts[0]
reference_path = (DATA_ROOT / train_df.loc[0, "image_path"])
with Image.open(reference_path) as reference_file:
    reference_image_size = reference_file.convert("RGB").size

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("Device:", device)
print("Data root:", DATA_ROOT)
print("Dataset run:", main_run_folder)
print("Reference image size:", reference_image_size)
print("Seed:", TEST5_SEED)
print("Batch size:", BATCH_SIZE)
print("Loss: cross-entropy")
print("Optimiser: SGD, learning rate", LEARNING_RATE, "momentum", MOMENTUM)
print("Maximum optimiser steps per stage:", TEST5_MAX_STEPS)
print("Augmentation: disabled, ToTensor only")
print()

path_checks = {
    "train manifest": TRAIN_CSV.is_file(),
    "sources manifest": SOURCES_CSV.is_file(),
    "diagnostic 4-bit train manifest": (DIAGNOSTIC_DATA_DIR / "train_4bit.csv").is_file(),
    "diagnostic 1-bit train manifest": (DIAGNOSTIC_DATA_DIR / "train_1bit.csv").is_file(),
}
print("Path checks:", path_checks)
if not all(path_checks.values()):
    raise RuntimeError("A required local path does not resolve: " + ", ".join(name for name, ok in path_checks.items() if not ok))
print()

rate_by_source = (
    sources_df[sources_df["split"].eq("train")]
    .drop_duplicates("source_id")
    .set_index("source_id")["payload_rate"]
    .astype(float)
    .to_dict()
)

selection_rng = np.random.default_rng(TEST5_SEED)
train_source_ids = train_df["source_id"].drop_duplicates().to_numpy()
test5_selected_ids = selection_rng.choice(train_source_ids, size=TEST5_SOURCE_COUNT, replace=False).tolist()
print("Selected training source IDs:", test5_selected_ids)
print()

preflight_rows = []
preflight_problems = []

for source_id in test5_selected_ids:
    source_rows = train_df[train_df["source_id"].eq(source_id)]
    if len(source_rows) != 2 or set(source_rows["label"]) != {0, 1}:
        preflight_problems.append(f"{source_id}: the training manifest does not have exactly one clean row and one stego row")
        continue

    clean_row = source_rows[source_rows["label"].eq(0)].iloc[0]
    stego_row = source_rows[source_rows["label"].eq(1)].iloc[0]

    if "/clean/" not in str(clean_row["image_path"]) or "/stego/" not in str(stego_row["image_path"]):
        preflight_problems.append(f"{source_id}: a training path does not point into the clean and stego folders")
        continue
    if "/val/" in str(clean_row["image_path"]) or "/test/" in str(clean_row["image_path"]):
        preflight_problems.append(f"{source_id}: a training path refers to validation or test data")
        continue
    if not str(clean_row["image_path"]).startswith(main_run_folder + "/"):
        preflight_problems.append(f"{source_id}: a training path is outside the main dataset run")

    with Image.open(DATA_ROOT / clean_row["image_path"]) as clean_file:
        clean_array = np.array(clean_file.convert("RGB"))
    with Image.open(DATA_ROOT / stego_row["image_path"]) as stego_file:
        stego_array = np.array(stego_file.convert("RGB"))

    if clean_array.shape != stego_array.shape:
        preflight_problems.append(f"{source_id}: clean and stego images have different shapes")
        continue

    changed_channels = int(np.count_nonzero(clean_array != stego_array))
    if changed_channels == 0:
        preflight_problems.append(f"{source_id}: the clean and stego images have no changed channel values")

    payload_rate = float(rate_by_source.get(source_id, stego_row["payload_rate"]))
    preflight_rows.append({
        "source_id": source_id,
        "payload_rate": round(payload_rate, 6),
        "changed_channels": changed_channels,
        "shape": list(clean_array.shape),
    })

print("Preflight table")
print(pd.DataFrame(preflight_rows).to_string(index=False))
print()

test5_preflight_by_source = {
    row["source_id"]: {
        "payload_rate": row["payload_rate"],
        "changed_channels": row["changed_channels"],
    }
    for row in preflight_rows
}

test5_preflight_result = {
    "seed": TEST5_SEED,
    "source_ids": test5_selected_ids,
    "per_source": preflight_rows,
    "runtime": {
        "python": sys.version.split()[0],
        "pytorch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "device": str(device),
        "data_root": str(DATA_ROOT),
        "dataset_run": main_run_folder,
        "reference_image_size": list(reference_image_size),
        "batch_size": BATCH_SIZE,
        "loss": "cross entropy",
        "optimiser": "SGD",
        "learning_rate": LEARNING_RATE,
        "momentum": MOMENTUM,
        "max_steps": TEST5_MAX_STEPS,
    },
    "path_checks": path_checks,
    "problems": preflight_problems,
}
save_result("test5_preflight", test5_preflight_result)

if preflight_problems:
    raise RuntimeError("Test 5 preflight found problems: " + "; ".join(preflight_problems))

print("Preflight gate passed. Every selected pair has a nonzero change count.")
print()


def run_paired_overfit_stage(stage_name, stage_ids, preflight_by_source):
    stage_rows = train_df[train_df["source_id"].isin(stage_ids)].copy()
    source_order = {source_id: position for position, source_id in enumerate(stage_ids)}
    stage_rows["source_order"] = stage_rows["source_id"].map(source_order)
    stage_rows = (
        stage_rows.sort_values(["source_order", "label"])
        .drop(columns="source_order")
        .reset_index(drop=True)
    )

    set_seed(TEST5_SEED)
    model = build_cnn_baseline().to(device)
    criterion = make_criterion()
    optimizer = make_optimizer(model)

    train_loader = DataLoader(
        LSBClassificationDataset(stage_rows, DATA_ROOT, make_eval_transform()),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )
    eval_loader = DataLoader(
        LSBClassificationDataset(stage_rows, DATA_ROOT, make_eval_transform()),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    history = []
    consecutive_checks = 0
    stop_reason = "step_cap_reached"
    best_eval_loss = None
    best_eval_loss_step = None
    best_eval_accuracy = -1.0
    best_eval_accuracy_step = None
    spike_steps = []
    nonfinite_steps = []

    for step in range(1, TEST5_MAX_STEPS + 1):
        train_loss, train_accuracy = train_epoch(model, train_loader, criterion, optimizer, device)
        eval_loss, eval_accuracy, eval_labels, eval_predictions = evaluate(model, eval_loader, criterion, device)

        train_loss_value = float(train_loss)
        eval_loss_value = float(eval_loss)
        train_loss_finite = bool(np.isfinite(train_loss_value))
        eval_loss_finite = bool(np.isfinite(eval_loss_value))

        if not (train_loss_finite and eval_loss_finite):
            nonfinite_steps.append(step)
        if (train_loss_finite and train_loss_value > TEST5_SPIKE_THRESHOLD) or (
            eval_loss_finite and eval_loss_value > TEST5_SPIKE_THRESHOLD
        ):
            spike_steps.append(step)

        confusion = torch.bincount(2 * eval_labels + eval_predictions, minlength=4).reshape(2, 2)

        history.append({
            "step": step,
            "train_loss": round(train_loss_value, 6) if train_loss_finite else None,
            "train_accuracy": round(float(train_accuracy), 6),
            "eval_loss": round(eval_loss_value, 6) if eval_loss_finite else None,
            "eval_accuracy": round(float(eval_accuracy), 6),
            "eval_prediction_counts": prediction_counts(eval_predictions),
            "eval_confusion_matrix": confusion.tolist(),
        })

        if best_eval_loss is None or (eval_loss_finite and eval_loss_value < best_eval_loss):
            best_eval_loss = round(eval_loss_value, 6) if eval_loss_finite else None
            best_eval_loss_step = step
        if eval_accuracy > best_eval_accuracy:
            best_eval_accuracy = round(float(eval_accuracy), 6)
            best_eval_accuracy_step = step

        if eval_accuracy >= 1.0 and eval_loss_finite and eval_loss_value < TEST5_EVAL_LOSS_TARGET:
            consecutive_checks += 1
        else:
            consecutive_checks = 0

        if step <= 5 or step % 5 == 0 or step in nonfinite_steps or step in spike_steps:
            print(
                f"step {step:03d} | train loss {history[-1]['train_loss']} | "
                f"train acc {history[-1]['train_accuracy']:.4f} | "
                f"eval loss {history[-1]['eval_loss']} | "
                f"eval acc {history[-1]['eval_accuracy']:.4f} | "
                f"eval preds {history[-1]['eval_prediction_counts']}"
            )

        if consecutive_checks >= TEST5_CONSECUTIVE_CHECKS:
            stop_reason = "memorised"
            print(
                f"Stage stopped at step {step}. The evaluation-mode memorisation criterion was met for "
                f"{TEST5_CONSECUTIVE_CHECKS} consecutive checks."
            )
            break

    final_row = history[-1]
    memorised = stop_reason == "memorised"

    return {
        "stage": stage_name,
        "source_ids": list(stage_ids),
        "source_count": len(stage_ids),
        "total_rows": int(len(stage_rows)),
        "clean_rows": int(stage_rows["label"].eq(0).sum()),
        "stego_rows": int(stage_rows["label"].eq(1).sum()),
        "payload_rates": {source_id: preflight_by_source[source_id]["payload_rate"] for source_id in stage_ids},
        "changed_channel_counts": {
            source_id: preflight_by_source[source_id]["changed_channels"] for source_id in stage_ids
        },
        "seed": TEST5_SEED,
        "batch_size": BATCH_SIZE,
        "max_steps": TEST5_MAX_STEPS,
        "steps_run": len(history),
        "history": history,
        "final_train_loss": final_row["train_loss"],
        "final_train_accuracy": final_row["train_accuracy"],
        "final_eval_loss": final_row["eval_loss"],
        "final_eval_accuracy": final_row["eval_accuracy"],
        "final_eval_prediction_counts": final_row["eval_prediction_counts"],
        "final_eval_confusion_matrix": final_row["eval_confusion_matrix"],
        "best_eval_loss": best_eval_loss,
        "best_eval_loss_step": best_eval_loss_step,
        "best_eval_accuracy": best_eval_accuracy,
        "best_eval_accuracy_step": best_eval_accuracy_step,
        "memorised": bool(memorised),
        "stop_reason": stop_reason,
        "loss_spike_steps": spike_steps,
        "nonfinite_steps": nonfinite_steps,
        "spike_threshold": TEST5_SPIKE_THRESHOLD,
        "memorisation_criterion": (
            "evaluation accuracy 1.0 and evaluation cross-entropy below 0.1 for "
            f"{TEST5_CONSECUTIVE_CHECKS} consecutive checks"
        ),
        "metric_timing": (
            "training metrics are from the forward pass before that step's weight update; "
            "evaluation metrics are measured with model.eval() after the update"
        ),
    }


test5_results = {}
print("Stage runner ready.")

Test 5. Paired clean/stego overfit
Preflight

Python: 3.13.7
PyTorch: 2.14.0+cu126 | CUDA available: True
Device: cuda
Data root: C:\Users\zhiho\Desktop\AI_stuff\05_Projects\School\Computer Vision and Deep Learning - Term Project\05_Data
Dataset run: lsb-run-20260922-233255-728056
Reference image size: (128, 128)
Seed: 20261001
Batch size: 64
Loss: cross-entropy
Optimiser: SGD, learning rate 0.01 momentum 0.9
Maximum optimiser steps per stage: 200
Augmentation: disabled, ToTensor only

Path checks: {'train manifest': True, 'sources manifest': True, 'diagnostic 4-bit train manifest': True, 'diagnostic 1-bit train manifest': True}

Selected training source IDs: ['55548', '44480', '12064', '28175', '30693', '22268', '20762', '03227', '29019', '42688', '15513', '26180', '49072', '38087', '09693', '44821']

Preflight table
source_id  payload_rate  changed_channels         shape
    55548      0.112872              2772 [128, 128, 3]
    44480      0.219749              5382 [128, 128, 3]
  

In [8]:
stage_name = "1 source"
test5_results[stage_name] = run_paired_overfit_stage(
    stage_name, test5_selected_ids[:1], test5_preflight_by_source
)
save_result("test5_paired_overfit", test5_results)

step 001 | train loss 0.703431 | train acc 0.5000 | eval loss 0.693342 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 2}
step 002 | train loss 1.190665 | train acc 0.5000 | eval loss 0.739072 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 2}
step 003 | train loss 6.630418 | train acc 0.5000 | eval loss 0.958858 | eval acc 0.5000 | eval preds {'clean': 2, 'stego': 0}
step 004 | train loss 8.750078 | train acc 0.5000 | eval loss 1.920345 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 2}
step 005 | train loss 10.148355 | train acc 0.5000 | eval loss 0.932246 | eval acc 0.5000 | eval preds {'clean': 2, 'stego': 0}
step 009 | train loss 3.931315 | train acc 0.5000 | eval loss 12128.076172 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 2}
step 010 | train loss 22.818035 | train acc 0.5000 | eval loss 339.254242 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 2}
step 011 | train loss 0.694598 | train acc 0.5000 | eval loss 1065.303833 | eval acc 0.5000 | eval pred

### Test 5 stage 1 observation (1 source, 2 rows)

Record the stop reason, the final and best evaluation-mode accuracy and loss, the prediction distribution, and whether the memorisation criterion was met. If it was not met within 200 steps, note the best observed metrics and any loss spikes.

In [9]:
stage_name = "4 sources"
test5_results[stage_name] = run_paired_overfit_stage(
    stage_name, test5_selected_ids[:4], test5_preflight_by_source
)
save_result("test5_paired_overfit", test5_results)

step 001 | train loss 0.711104 | train acc 0.5000 | eval loss 0.6932 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 8}
step 002 | train loss 1.306785 | train acc 0.5000 | eval loss 0.748533 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 8}
step 003 | train loss 6.340966 | train acc 0.5000 | eval loss 1.230047 | eval acc 0.5000 | eval preds {'clean': 8, 'stego': 0}
step 004 | train loss 13.899293 | train acc 0.5000 | eval loss 2.148882 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 8}
step 005 | train loss 12.10676 | train acc 0.5000 | eval loss 0.693147 | eval acc 0.5000 | eval preds {'clean': 6, 'stego': 2}
step 010 | train loss 0.693745 | train acc 0.5000 | eval loss 0.693735 | eval acc 0.5000 | eval preds {'clean': 6, 'stego': 2}
step 015 | train loss 0.693526 | train acc 0.5000 | eval loss 0.693506 | eval acc 0.5000 | eval preds {'clean': 2, 'stego': 6}
step 020 | train loss 0.693388 | train acc 0.5000 | eval loss 0.693348 | eval acc 0.5000 | eval preds {'clean': 

### Test 5 stage 2 observation (4 sources, 8 rows)

Record the stop reason, the final and best evaluation-mode accuracy and loss, the prediction distribution, and the comparison with the 1-source stage.

In [10]:
stage_name = "16 sources"
test5_results[stage_name] = run_paired_overfit_stage(
    stage_name, test5_selected_ids[:16], test5_preflight_by_source
)
save_result("test5_paired_overfit", test5_results)

step 001 | train loss 0.694867 | train acc 0.5312 | eval loss 0.693554 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 32}
step 002 | train loss 0.785277 | train acc 0.5000 | eval loss 0.771871 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 32}
step 003 | train loss 7.17189 | train acc 0.5000 | eval loss 0.69394 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 32}
step 004 | train loss 0.693875 | train acc 0.5000 | eval loss 0.693578 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 32}
step 005 | train loss 0.693363 | train acc 0.5000 | eval loss 0.693239 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 32}
step 010 | train loss 0.693487 | train acc 0.5000 | eval loss 0.693485 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 32}
step 015 | train loss 0.693356 | train acc 0.5000 | eval loss 0.69333 | eval acc 0.5000 | eval preds {'clean': 0, 'stego': 32}
step 020 | train loss 0.693248 | train acc 0.5000 | eval loss 0.693233 | eval acc 0.5000 | eval preds {'cle

### Test 5 stage 3 observation (16 sources, 32 rows)

Record the stop reason, the final and best evaluation-mode accuracy and loss, the prediction distribution, and how ability changes from 1 to 4 to 16 sources. A 200-step cap is not proof that this subset cannot be fitted.

## Test 6. Pixel-level difference audit

Purpose. Quantify the actual pixel differences between matched clean and stego images before attributing the baseline failure only to a weak signal. The audit compares the main 1-bit training pairs, the diagnostic matched 1-bit control pairs, and the diagnostic 4-bit pairs.

Sample. At least 30 shared training sources are drawn across low, middle, and high payload-rate bands, and every Test 5 source is added. Differences are computed as signed integer subtractions on RGB arrays. The recorded payload rate is the fraction of channel positions selected for embedding, not the fraction of positions whose value actually changed.

Checks. Main 1-bit pairs should only ever differ by exactly 1. The diagnostic clean copies must be identical to the main clean images. The 4-bit images should change more channel values and produce larger differences than their matched 1-bit controls. A sample audit does not validate every generated image.

In [11]:
print("Test 6. Pixel-level difference audit")
print()

TEST6_SEED = 20261002
TEST6_SAMPLE_PER_BAND = 10
TEST6_MIN_SHARED_SOURCES = 30
TEST6_RATE_BANDS = (
    ("low", 0.0, 0.35),
    ("middle", 0.35, 0.65),
    ("high", 0.65, 1.01),
)

diagnostic_train_4bit = pd.read_csv(DIAGNOSTIC_DATA_DIR / "train_4bit.csv", dtype=SOURCE_DTYPE_MAP)
diagnostic_train_1bit = pd.read_csv(DIAGNOSTIC_DATA_DIR / "train_1bit.csv", dtype=SOURCE_DTYPE_MAP)

shared_sources = sorted(
    set(diagnostic_train_4bit["source_id"]) & set(diagnostic_train_1bit["source_id"]) & set(train_df["source_id"])
)
print("Shared training sources available for the audit:", len(shared_sources))

audit_problems = []

for condition_name, manifest in (
    ("diagnostic 4-bit", diagnostic_train_4bit),
    ("diagnostic 1-bit", diagnostic_train_1bit),
):
    if manifest["source_id"].duplicated().any():
        audit_problems.append(f"{condition_name} manifest has duplicate source rows")

sampled_shared_ids = []
sampling_rng = np.random.default_rng(TEST6_SEED)
for band_name, low, high in TEST6_RATE_BANDS:
    band_ids = sorted(source_id for source_id in shared_sources if low <= rate_by_source[source_id] < high)
    if len(band_ids) > TEST6_SAMPLE_PER_BAND:
        chosen = sampling_rng.choice(band_ids, size=TEST6_SAMPLE_PER_BAND, replace=False).tolist()
    else:
        chosen = band_ids
    sampled_shared_ids.extend(chosen)
    print(f"band {band_name}: {len(band_ids)} available, {len(chosen)} sampled")

missing_shared = TEST6_MIN_SHARED_SOURCES - len(sampled_shared_ids)
remaining_shared = sorted(set(shared_sources) - set(sampled_shared_ids))
if missing_shared > 0 and remaining_shared:
    extra_shared = sampling_rng.choice(
        remaining_shared, size=min(missing_shared, len(remaining_shared)), replace=False
    ).tolist()
    sampled_shared_ids.extend(extra_shared)
    print(f"top-up sampling added {len(extra_shared)} shared sources")

audit_ids = list(dict.fromkeys(sampled_shared_ids + list(test5_selected_ids)))
print("Audited sources:", len(audit_ids), "| sampled shared:", len(sampled_shared_ids), "| Test 5 sources added:", len(set(test5_selected_ids) - set(sampled_shared_ids)))
print()


def rate_band(payload_rate):
    if payload_rate < 0.35:
        return "low"
    if payload_rate < 0.65:
        return "middle"
    return "high"


def load_rgb(image_path):
    with Image.open(DATA_ROOT / image_path) as image_file:
        return np.array(image_file.convert("RGB"))


def difference_metrics(source_id, condition, payload_rate, clean_array, stego_array):
    clean_int = clean_array.astype(np.int16)
    stego_int = stego_array.astype(np.int16)
    difference = stego_int - clean_int
    absolute = np.abs(difference)
    changed_mask = absolute > 0
    changed = int(changed_mask.sum())
    total = int(absolute.size)
    changed_values = absolute[changed_mask]

    histogram_values, histogram_counts = np.unique(absolute[changed_mask], return_counts=True)
    histogram = {str(int(value)): int(count) for value, count in zip(histogram_values, histogram_counts)}

    return {
        "source_id": source_id,
        "condition": condition,
        "payload_rate": round(float(payload_rate), 6),
        "rate_band": rate_band(float(payload_rate)),
        "height": int(clean_array.shape[0]),
        "width": int(clean_array.shape[1]),
        "channels": int(clean_array.shape[2]),
        "total_channel_values": total,
        "unchanged": int(total - changed),
        "changed": changed,
        "changed_percent": round(100 * changed / total, 6),
        "signed_min": int(difference.min()),
        "signed_max": int(difference.max()),
        "changed_by_one": int((absolute == 1).sum()),
        "changed_by_more_than_one": int((absolute > 1).sum()),
        "abs_difference_histogram": histogram,
        "typical_abs_difference_changed": round(float(changed_values.mean()), 6) if changed else None,
        "median_abs_difference_changed": round(float(np.median(changed_values)), 6) if changed else None,
        "typical_abs_difference_all": round(float(absolute.mean()), 6),
    }


diag_4bit_clean = diagnostic_train_4bit[diagnostic_train_4bit["label"].eq(0)].drop_duplicates("source_id").set_index("source_id")
diag_4bit_stego = diagnostic_train_4bit[diagnostic_train_4bit["label"].eq(1)].drop_duplicates("source_id").set_index("source_id")
diag_1bit_stego = diagnostic_train_1bit[diagnostic_train_1bit["label"].eq(1)].drop_duplicates("source_id").set_index("source_id")

pair_rows = []
for source_id in audit_ids:
    main_rows = train_df[train_df["source_id"].eq(source_id)]
    clean_rows = main_rows[main_rows["label"].eq(0)]
    stego_rows = main_rows[main_rows["label"].eq(1)]

    if len(clean_rows) != 1 or len(stego_rows) != 1:
        audit_problems.append(f"{source_id}: the main manifest does not have exactly one clean row and one stego row")
        continue

    payload_rate = float(rate_by_source[source_id])
    main_clean = load_rgb(clean_rows.iloc[0]["image_path"])
    main_stego = load_rgb(stego_rows.iloc[0]["image_path"])
    if main_clean.shape != main_stego.shape:
        audit_problems.append(f"{source_id}: the main clean and stego images have different shapes")
        continue
    main_metrics = difference_metrics(source_id, "main_1bit", payload_rate, main_clean, main_stego)
    pair_rows.append(main_metrics)

    if main_metrics["changed"] == 0:
        audit_problems.append(f"{source_id}: the main 1-bit pair has no changed channel values")
    if main_metrics["changed_by_more_than_one"] > 0:
        audit_problems.append(
            f"{source_id}: the main 1-bit pair has {main_metrics['changed_by_more_than_one']} channel values changed by more than 1"
        )

    if source_id in shared_sources:
        diagnostic_clean = load_rgb(diag_4bit_clean.loc[source_id, "image_path"])
        diagnostic_one_bit = load_rgb(diag_1bit_stego.loc[source_id, "image_path"])
        diagnostic_four_bit = load_rgb(diag_4bit_stego.loc[source_id, "image_path"])

        if not np.array_equal(main_clean, diagnostic_clean):
            audit_problems.append(f"{source_id}: the diagnostic clean copy is not identical to the main clean image")
        if diagnostic_one_bit.shape != diagnostic_clean.shape or diagnostic_four_bit.shape != diagnostic_clean.shape:
            audit_problems.append(f"{source_id}: diagnostic images have different shapes")
            continue

        pair_rows.append(difference_metrics(source_id, "diagnostic_1bit", payload_rate, diagnostic_clean, diagnostic_one_bit))
        pair_rows.append(difference_metrics(source_id, "diagnostic_4bit", payload_rate, diagnostic_clean, diagnostic_four_bit))

if not pair_rows:
    raise RuntimeError("The pixel audit found no comparable pairs.")

pair_df = pd.DataFrame(pair_rows)


def mean_or_none(series):
    values = series.dropna()
    return round(float(values.mean()), 6) if len(values) else None


condition_summaries = {}
for condition_name, group in pair_df.groupby("condition", sort=True):
    condition_summaries[condition_name] = {
        "pairs": int(len(group)),
        "mean_changed_percent": round(float(group["changed_percent"].mean()), 6),
        "min_changed_percent": round(float(group["changed_percent"].min()), 6),
        "max_changed_percent": round(float(group["changed_percent"].max()), 6),
        "mean_typical_abs_difference_changed": mean_or_none(group["typical_abs_difference_changed"]),
        "min_typical_abs_difference_changed": (
            round(float(group["typical_abs_difference_changed"].min()), 6)
            if group["typical_abs_difference_changed"].notna().any()
            else None
        ),
        "max_typical_abs_difference_changed": (
            round(float(group["typical_abs_difference_changed"].max()), 6)
            if group["typical_abs_difference_changed"].notna().any()
            else None
        ),
        "mean_typical_abs_difference_all": round(float(group["typical_abs_difference_all"].mean()), 6),
        "mean_changed_by_one": round(float(group["changed_by_one"].mean()), 6),
        "mean_changed_by_more_than_one": round(float(group["changed_by_more_than_one"].mean()), 6),
        "pairs_with_only_one_level_changes": int(group["changed_by_more_than_one"].eq(0).sum()),
    }

band_summaries = []
for (condition_name, band_name), group in pair_df.groupby(["condition", "rate_band"], sort=True):
    band_summaries.append({
        "condition": condition_name,
        "rate_band": band_name,
        "pairs": int(len(group)),
        "mean_payload_rate": round(float(group["payload_rate"].mean()), 6),
        "mean_changed_percent": round(float(group["changed_percent"].mean()), 6),
        "mean_typical_abs_difference_changed": mean_or_none(group["typical_abs_difference_changed"]),
    })

shared_pair_df = pair_df[pair_df["source_id"].isin(shared_sources)]
one_bit_group = shared_pair_df[shared_pair_df["condition"].eq("diagnostic_1bit")]
four_bit_group = shared_pair_df[shared_pair_df["condition"].eq("diagnostic_4bit")]
depth_table = one_bit_group.set_index("source_id")[["changed_percent", "typical_abs_difference_changed"]].join(
    four_bit_group.set_index("source_id")[["changed_percent", "typical_abs_difference_changed"]],
    lsuffix="_1bit",
    rsuffix="_4bit",
)

depth_comparison = {
    "shared_pairs": int(len(depth_table)),
    "mean_changed_percent_1bit": round(float(depth_table["changed_percent_1bit"].mean()), 6),
    "mean_changed_percent_4bit": round(float(depth_table["changed_percent_4bit"].mean()), 6),
    "mean_typical_abs_difference_1bit": mean_or_none(depth_table["typical_abs_difference_changed_1bit"]),
    "mean_typical_abs_difference_4bit": mean_or_none(depth_table["typical_abs_difference_changed_4bit"]),
    "pairs_with_more_changes_in_4bit": int((depth_table["changed_percent_4bit"] > depth_table["changed_percent_1bit"]).sum()),
    "all_matched_pairs_have_more_changes_in_4bit": bool(
        (depth_table["changed_percent_4bit"] > depth_table["changed_percent_1bit"]).all()
    ),
    "all_matched_pairs_have_larger_differences_in_4bit": bool(
        (depth_table["typical_abs_difference_changed_4bit"] > depth_table["typical_abs_difference_changed_1bit"]).all()
    ),
}

print("Condition summaries")
print(pd.DataFrame([{"condition": name, **summary} for name, summary in condition_summaries.items()]).to_string(index=False))
print()
print("Changed percentage by payload-rate band")
print(pd.DataFrame(band_summaries).to_string(index=False))
print()
print("Matched depth comparison")
for key, value in depth_comparison.items():
    print(" ", key + ":", value)
print()
print("Data issues flagged by the audit:", len(audit_problems))
for problem in audit_problems[:10]:
    print(" -", problem)
print()

test6_result = {
    "seed": TEST6_SEED,
    "sample_per_band": TEST6_SAMPLE_PER_BAND,
    "rate_bands": [{"name": name, "low": low, "high": high} for name, low, high in TEST6_RATE_BANDS],
    "selection_rule": (
        "stratified sample of shared training sources across low, middle, and high recorded payload-rate bands, "
        "plus every Test 5 source"
    ),
    "shared_sources_available": int(len(shared_sources)),
    "sampled_shared_sources": sampled_shared_ids,
    "test5_sources": list(test5_selected_ids),
    "audited_sources": audit_ids,
    "pairs": pair_rows,
    "condition_summaries": condition_summaries,
    "band_summaries": band_summaries,
    "depth_comparison": depth_comparison,
    "problems": audit_problems,
    "difference_method": "signed int16 subtraction on RGB arrays",
    "sample_note": "This sample audit does not validate every generated image.",
}
save_result("test6_pixel_difference_audit", test6_result)

pair_csv = pair_df.drop(columns=["abs_difference_histogram"])
pair_csv_path = RESULTS_DIR / "test6_pixel_difference_pairs.csv"
pair_csv.to_csv(pair_csv_path, index=False)
print("Saved per-pair rows to", pair_csv_path)

Test 6. Pixel-level difference audit

Shared training sources available for the audit: 2000
band low: 677 available, 10 sampled
band middle: 792 available, 10 sampled
band high: 531 available, 10 sampled
Audited sources: 46 | sampled shared: 30 | Test 5 sources added: 16

Condition summaries
      condition  pairs  mean_changed_percent  min_changed_percent  max_changed_percent  mean_typical_abs_difference_changed  min_typical_abs_difference_changed  max_typical_abs_difference_changed  mean_typical_abs_difference_all  mean_changed_by_one  mean_changed_by_more_than_one  pairs_with_only_one_level_changes
diagnostic_1bit     31             24.005258             7.031250            41.432699                             1.000000                             1.00000                            1.000000                         0.240053         11799.064516                       0.000000                                 31
diagnostic_4bit     31             44.992361            13.179525          

### Test 6 observation (record after running)

Record what the main 1-bit differences look like, whether every changed value differs by exactly 1, how the changed percentage relates to the recorded payload rate, how much stronger the 4-bit condition is than its matched 1-bit control, and whether any data problems were flagged. State that this sample audit does not validate every generated image.

## Combined diagnosis (Tests 1 to 6)

Purpose. Combine the earlier four tests with the paired overfit test and the pixel audit into one evidence-based draft. This section supersedes the preliminary diagnosis.

In [14]:
print("Combined diagnosis (Tests 1 to 6)")
print()

test1_result = load_result("test1_dataset_sanity")
test2_result = load_result("test2_overfit_subset")
test3_result = load_result("test3_augmentation_comparison")
test4_result = load_result("test4_strength_sanity")
test5_result = load_result("test5_paired_overfit")
test6_result = load_result("test6_pixel_difference_audit")

print("Earlier tests 1 to 4")
print(" Test 1 status:", test1_result["status"])
print(" Test 2 unpaired overfit memorised:", test2_result["memorised"], "in", test2_result["optimiser_steps"], "steps")
print(
    " Test 3 final validation accuracy | augmented:",
    test3_result["augmented"]["final_val_accuracy"],
    "| no augmentation:",
    test3_result["no_augmentation"]["final_val_accuracy"],
)
print(
    " Test 4 final validation accuracy | 4-bit:",
    test4_result["conditions"]["4-bit"]["final_val_accuracy"],
    "| matched 1-bit:",
    test4_result["conditions"]["1-bit control"]["final_val_accuracy"],
)
print()

print("Training mode, paired stages")
for stage_name, stage in test5_result.items():
    best_train_accuracy = max(row["train_accuracy"] for row in stage["history"])
    print(
        f" {stage_name}: {stage['source_count']} sources, {stage['total_rows']} rows, "
        f"{stage['steps_run']} steps, final train accuracy {stage['final_train_accuracy']:.4f}, "
        f"best train accuracy {best_train_accuracy:.4f}"
    )
print()

print("Evaluation mode, paired stages")
stage_table = pd.DataFrame([
    {
        "stage": stage_name,
        "rows": stage["total_rows"],
        "steps_run": stage["steps_run"],
        "final_eval_accuracy": stage["final_eval_accuracy"],
        "final_eval_loss": stage["final_eval_loss"],
        "best_eval_accuracy": stage["best_eval_accuracy"],
        "memorised": stage["memorised"],
        "stop_reason": stage["stop_reason"],
        "spike_steps": len(stage["loss_spike_steps"]),
        "nonfinite_steps": len(stage["nonfinite_steps"]),
    }
    for stage_name, stage in test5_result.items()
])
print(stage_table.to_string(index=False))
print()

print("Main 1-bit pixel differences")
main_summary = test6_result["condition_summaries"].get("main_1bit", {})
print(" pairs audited:", main_summary.get("pairs"))
print(" mean changed percent:", main_summary.get("mean_changed_percent"))
print(" min changed percent:", main_summary.get("min_changed_percent"))
print(" max changed percent:", main_summary.get("max_changed_percent"))
print(" mean typical absolute difference among changed values:", main_summary.get("mean_typical_abs_difference_changed"))
print(" pairs where every changed value differs by exactly 1:", main_summary.get("pairs_with_only_one_level_changes"))
print()

print("Matched diagnostic depth comparison")
depth = test6_result["depth_comparison"]
print(" shared pairs:", depth["shared_pairs"])
print(" mean changed percent | 1-bit control:", depth["mean_changed_percent_1bit"], "| 4-bit:", depth["mean_changed_percent_4bit"])
print(
    " mean typical absolute difference | 1-bit control:",
    depth["mean_typical_abs_difference_1bit"],
    "| 4-bit:",
    depth["mean_typical_abs_difference_4bit"],
)
print(" every matched pair has more changed values in 4-bit:", depth["all_matched_pairs_have_more_changes_in_4bit"])
print(" every matched pair has larger typical differences in 4-bit:", depth["all_matched_pairs_have_larger_differences_in_4bit"])
print()

print("Data issues flagged by the audit:", len(test6_result["problems"]))
for problem in test6_result["problems"][:10]:
    print(" -", problem)
print()

draft_findings = []

if test6_result["problems"]:
    draft_findings.append("The pixel audit flagged data issues. Investigate data generation before any model change.")
else:
    draft_findings.append("The pixel audit found the expected differences, so the sampled pixel data passed its checks.")

memorised_stages = [stage_name for stage_name, stage in test5_result.items() if stage["memorised"]]
if not memorised_stages:
    draft_findings.append("No paired stage reached the evaluation-mode memorisation criterion within the step cap.")
elif len(memorised_stages) == len(test5_result):
    draft_findings.append("Every paired stage memorised in evaluation mode, so the CNN can fit matched clean and stego pairs at these sizes.")
else:
    draft_findings.append(
        f"Memorisation was reached for {', '.join(memorised_stages)} but not for every stage, "
        "so the ability changes with the number of paired sources."
    )

train_high_eval_low = [
    stage_name
    for stage_name, stage in test5_result.items()
    if max(row["train_accuracy"] for row in stage["history"]) >= 0.9 and stage["best_eval_accuracy"] < 0.9
]
spike_stages = [
    stage_name
    for stage_name, stage in test5_result.items()
    if stage["loss_spike_steps"] or stage["nonfinite_steps"]
]

if train_high_eval_low:
    draft_findings.append(
        "Training-mode accuracy is high while evaluation-mode accuracy stays low in "
        + ", ".join(train_high_eval_low)
        + ". Inspect BatchNorm train and evaluation behaviour next."
    )
elif spike_stages:
    draft_findings.append(
        "Loss spikes or non-finite losses appear in "
        + ", ".join(spike_stages)
        + ". Investigate optimisation stability before changing the input representation."
    )
elif memorised_stages and len(memorised_stages) < len(test5_result):
    draft_findings.append(
        "The paired fit works for smaller subsets but not the largest. The remaining question is learning "
        "and generalisation at scale, so a controlled representation experiment is justified next."
    )
elif memorised_stages:
    draft_findings.append(
        "The paired fit works while Tests 3 and 4 failed at scale. The remaining question is learning and "
        "generalisation at scale, so a controlled representation experiment is justified next."
    )
else:
    draft_findings.append(
        "The paired fit failed despite the verified pixel differences. Investigate training behaviour before "
        "changing the input representation."
    )

print("Draft combined findings")
for finding in draft_findings:
    print("-", finding)
print()
print("Confirm or amend these findings in the final diagnosis cell.")

Combined diagnosis (Tests 1 to 6)

Earlier tests 1 to 4
 Test 1 status: passed
 Test 2 unpaired overfit memorised: True in 21 steps
 Test 3 final validation accuracy | augmented: 0.5 | no augmentation: 0.5
 Test 4 final validation accuracy | 4-bit: 0.5 | matched 1-bit: 0.5

Training mode, paired stages
 1 source: 1 sources, 2 rows, 200 steps, final train accuracy 0.5000, best train accuracy 0.5000
 4 sources: 4 sources, 8 rows, 200 steps, final train accuracy 0.5000, best train accuracy 0.6250
 16 sources: 16 sources, 32 rows, 200 steps, final train accuracy 0.5000, best train accuracy 0.5312

Evaluation mode, paired stages
     stage  rows  steps_run  final_eval_accuracy  final_eval_loss  best_eval_accuracy  memorised      stop_reason  spike_steps  nonfinite_steps
  1 source     2        200                  0.5         0.693147               0.500      False step_cap_reached           12                0
 4 sources     8        200                  0.5         0.693082               

## Final diagnosis (confirmed from the recorded results)

Confirmed against the saved records in `diagnostic_results/`. The numbers below come from those records.

- Most likely failure category: the raw RGB representation gives this CNN no usable signal for the 1-bit task. The failure is not in the data, the labels, or the ability of the model to fit examples.
- Evidence:
  - The dataset and label checks passed. The manifests are balanced, the image paths resolve, and no source identity crosses a split.
  - The model memorised 32 unrelated training images in 21 steps, so the training loop, the gradients, the optimiser, and the evaluation path all work.
  - Matched clean and stego images from the same source did not memorise in 200 steps at 1, 4, or 16 sources. Training-mode accuracy also stayed at 0.50, so the model never separated a paired difference, not even in training mode.
  - Removing augmentation changed nothing, and the matched 4-bit condition, which carries a much stronger and more widespread change, also stayed at 0.50.
  - The pixel audit confirms the data behaves as generated. All 46 audited 1-bit pairs changed by exactly one intensity level, with a mean of 24.6 percent of channels changed. The matched 4-bit images changed more, with a mean of 45.0 percent of channels and a typical difference of 5.69.
  - Loss spikes appeared in the 1-source and 4-source paired stages (12 and 2 steps). Optimisation stability is therefore a second, smaller concern to watch, not the primary explanation.
- Next justified action: change the input representation in `cnn_baselineV2`, starting with a fixed high-pass or noise-residual input, and record loss and gradient finiteness while training.
- Should `cnn_baselineV2` proceed to input-representation changes: yes. Failure on raw RGB, including on the stronger 4-bit data, supports that. The paired test alone does not prove a residual filter will work, so the change still has to be measured against the raw control.

The separate diagnostic manifests still report duplicate source rows. Confirm their intended structure before using them again.
